## Analisi disomogeneità
Fitto prima la retta dei residui vedo l'andamento e provo a trovare un fit che lo soddisfi cosi uso quest'ultimo per vedere se poi l'errore sistematico va via e i residui sono governati dal rumore o errori casuali

In [31]:
from IPython.display import display, Image
import math
import numpy as np
import pandas as pd
import ROOT


file = "Esperienza4.xlsx"


df = pd.read_excel(file, sheet_name="Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype='float64')

x = df[:, 1]         
y = df[:, 2]          
err_x = df[:, 3]
err_y = df[:, 4]

indici = [0, 1, 2, 3, 4, 5]
x = np.delete(x, indici)
y = np.delete(y, indici)
err_x = np.delete(err_x, indici)
err_y = np.delete(err_y, indici)

n = len(x)                
zeri = np.zeros(n)         


formula_malus = "([0]*pow(cos(x-[2]),2) + [1]) * (1 + [3]*x)"

f1 = ROOT.TF1("f1", formula_malus, min(x), max(x))
f1.SetParameters(y.max() - y.min(), y.min(), x[np.argmax(y)], 0)
f1.SetParNames("A", "B", "C", "D")

grafico = ROOT.TGraphErrors(n, x, y, err_x, err_y)
fit1 = grafico.Fit(f1, "S")          

print(" Malus + deriva")
print(f"   chi2 = {fit1.Chi2():.0f}  su {fit1.Ndf()} gradi di libertà")
print("   (se chi2/ndf è molto maggiore di 1, i residui non sono solo rumore)\n")



# Residui e errore efficace di ogni punto
residui1 = []
errori1 = []
for i in range(n):
    pendenza = f1.Derivative(x[i])                                  
    errore = math.sqrt(err_y[i] ** 2 + (pendenza * err_x[i]) ** 2)  
    residui1.append(y[i] - f1.Eval(x[i]))                            
    errori1.append(errore)
residui1 = np.array(residui1)
errori1 = np.array(errori1)


# Per k = 1, 2... provo a descrivere i residui
# se l'onda è grande quel k è presente nei residui
grafico_residui = ROOT.TGraphErrors(n, x, residui1, zeri, errori1)

ampiezze = []
fasi = []
print(" ampiezza dell'onda nei residui, per k gobbe per giro")
for k in range(1, 7):
    formula_k = "[0]*cos(" + str(k) + "*x) + [1]*sin(" + str(k) + "*x)"
    onda_k = ROOT.TF1("onda_" + str(k), formula_k, min(x), max(x))
    onda_k.SetParameters(0.01, 0.01)
    grafico_residui.Fit(onda_k, "SQ0")

    a = onda_k.GetParameter(0)
    b = onda_k.GetParameter(1)
    ampiezze.append(math.sqrt(a ** 2 + b ** 2))    
    fasi.append(math.atan2(b, a))                  
    print(f"   k = {k}:  {ampiezze[-1]:.4f} V")

K = ampiezze.index(max(ampiezze)) + 1        # k con l'onda più grande


print(f"   -> domina k = {K}: periodo 360°/{K} = {360 / K:.0f}°")
print("      (k = 2 vale ~0 per costruzione: il fit di Malus l'ha già assorbita)\n")


# aggiungo l'onda alla funzione di fit e rifitto
formula_onda = formula_malus + " + [4]*cos(" + str(K) + "*x-[5])"

f2 = ROOT.TF1("f2", formula_onda, min(x), max(x))
f2.SetParameters(f1.GetParameter(0), f1.GetParameter(1), f1.GetParameter(2), f1.GetParameter(3), ampiezze[K - 1], fasi[K - 1])      
f2.SetParNames("A", "B", "C", "D", "G", "H")

fit2 = grafico.Fit(f2, "SQ0")

A = f2.GetParameter(0)
B = f2.GetParameter(1)
G = f2.GetParameter(4)
H = f2.GetParameter(5)
err_G = f2.GetParError(4)

print(" Malus + deriva + onda")
print(f"   chi2 = {fit2.Chi2():.0f}  su {fit2.Ndf()} gradi di libertà")
print(f"   ampiezza dell'onda G = {abs(G):.4f} ± {err_G:.4f} V  =  {abs(G) / (A + B) * 100:.2f} % di Vmax = {A + B:.2f} V\n")

# Nuovi residui
residui2 = []
errori2 = []
for i in range(n):
    pendenza = f2.Derivative(x[i])
    errore = math.sqrt(err_y[i] ** 2 + (pendenza * err_x[i]) ** 2)
    residui2.append(y[i] - f2.Eval(x[i]))
    errori2.append(errore)
residui2 = np.array(residui2)
errori2 = np.array(errori2)


print(" confronto")
print(f"   chi2 prima = {fit1.Chi2():.0f},   chi2 dopo = {fit2.Chi2():.0f}")
print("   Se i residui ora sono rumore, nel grafico (3) non si vedono più onde.")

#  GRAFICI


# residui di Malus + deriva, con sopra l'onda trovata
canvas1 = ROOT.TCanvas("canvas1", "Residui iniziali", 800, 600)
g1 = ROOT.TGraphErrors(n, x, residui1, zeri, errori1)
g1.SetTitle("(1) residui di Malus + deriva;#theta [rad];V - V_{fit} [V]")
g1.SetMarkerStyle(20)
g1.SetMarkerSize(0.5)
g1.Draw("AP")

onda_trovata = ROOT.TF1("onda_trovata", "[0]*cos(" + str(K) + "*x-[1])", min(x), max(x))
onda_trovata.SetParameters(G, H)
onda_trovata.SetLineColor(ROOT.kRed)
onda_trovata.Draw("same")

# ampiezza di ogni onda
canvas2 = ROOT.TCanvas("canvas2", "Ampiezze", 800, 600)
istogramma = ROOT.TH1F("istogramma", "(2) quali onde ci sono nei residui;k = gobbe per giro;ampiezza [V]", 6, 0.5, 6.5)
for k in range(1, 7):
    istogramma.SetBinContent(k, ampiezze[k - 1])
istogramma.SetFillColor(ROOT.kAzure + 1)
istogramma.SetStats(0)
istogramma.Draw("hist")

# residui dopo aver aggiunto l'onda (divisi per il loro errore)
canvas3 = ROOT.TCanvas("canvas3", "Residui finali", 800, 600)
g3 = ROOT.TGraphErrors(n, x, residui2 / errori2, zeri, np.ones(n))
g3.SetTitle("(3) residui dopo aver aggiunto l'onda;#theta [rad];residuo normalizzato")
g3.SetMarkerStyle(20)
g3.SetMarkerSize(0.5)
g3.SetMarkerColor(ROOT.kGreen + 2)
g3.Draw("AP")

canvas1.Update()
canvas1.SaveAs("grafico1.png")

canvas2.Update()
canvas2.SaveAs("grafico2.png")

canvas3.Update()
canvas3.SaveAs("grafico3.png")

 Malus + deriva
   chi2 = 652  su 190 gradi di libertà
   (se chi2/ndf è molto maggiore di 1, i residui non sono solo rumore)

 ampiezza dell'onda nei residui, per k gobbe per giro
   k = 1:  0.0072 V
   k = 2:  0.0007 V
   k = 3:  0.0071 V
   k = 4:  0.0353 V
   k = 5:  0.0019 V
   k = 6:  0.0017 V
   -> domina k = 4: periodo 360°/4 = 90°
      (k = 2 vale ~0 per costruzione: il fit di Malus l'ha già assorbita)

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      651.702
NDf                       =          190
Edm                       =   8.1739e-09
NCalls                    =          110
A                         =      1.99396   +/-   0.00358707  
B                         =      2.80099   +/-   0.00232427  
C                         =      3.11989   +/-   0.00103293  
D                         =   -0.0015975   +/-   0.000159403 
 Malus + deriva + onda
   chi2 = 101  su 188 gradi di libertà
   ampiezza dell'onda G = 0.0416 ± 0.0

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas1
Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas2
Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
Warning in <TROOT::Append>: Replacing existing TH1: istogramma (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas3
Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
Error in <TASImage::WriteImage>: no image in memory. Draw something first
Error in <TASImage::WriteImage>: no image in memory. Draw something first
Error in <TASImage::WriteImage>: no image in memory. Draw something first
Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
Error in <DrawPolyLineAux>: invalid pad's geometry
Error in <DrawPolyLineAux>: invalid pad's geometry
Error in <DrawPolyLineAux>: invalid pad's geometry


Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
Error in <DrawFillAreaAux>: invalid pad's geometry
Error in <DrawPolyLineAux>: invalid pad's geometry


Warning in <TCanvas::ResizePad>: gPad has at least one zero dimension.
